In [85]:
# import libraries
import pandas as pd
import numpy as np
import ast
import torch

import torch.nn as nn
from sklearn.model_selection import train_test_split

In [86]:
# define file paths 
audio_path = "./dataset/audio_features.csv"
lb_path = "./dataset/leader_board.csv"

# load the datasets into pandas DataFrames
audio = pd.read_csv(audio_path)
lb = pd.read_csv(lb_path)

# Predicting Whether a Song is a "Hit"

### Data Overview:
The Billboard Hot 100 Weekly Charts with Audio dataset combines historical data from the Billboard Hot 100 weekly singles chart with detailed audio features extracted from Spotify (from 05-01-58 to 05-28-2021).

---

# Dataset 1: hotstuff.csv

Contains each song’s performance on the weekly Billboard Hot 100 chart. It includes details such as how many times a song appears on the chart, its position in the previous week, its peak position (the highest rank it achieved), and the total number of weeks it remained on the chart. 

In [87]:
# Convert WeekID to datetime
lb["WeekID"] = pd.to_datetime(lb["WeekID"], format="%m/%d/%Y", errors="coerce")

# Aggregate weekly chart data to the song level
lb_song = (
    lb.groupby("SongID", as_index=False)
      .agg(
          # First and last week the song appeared on the chart
          first_week=("WeekID", "min"),
          last_week=("WeekID", "max"),
          
          # Best (highest) chart position achieved
          peak_position=("Peak Position", "min"),

          # Total number of weeks the song appeared on the Billboardd Hot 100 chart.
          weeks_on_chart=("Weeks on Chart", "max"),
      )
      .assign(
          # Defining the hit: 
          # Peak Position <= 10
          # Weeks on the chart >= 15
          hit=lambda d: ((d.peak_position <= 10) & (d.weeks_on_chart >= 15)).astype(int)
      )
)

---

## Dataset 2:

The Hot 100 Audio Features dataset contains Spotify-derived audio and metadata for songs appearing on the Billboard Hot 100 chart. It includes information on song and artist identifiers, genre, duration, explicit content, album information, and core audio characteristics such as danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, and time signature. For this analysis, we retained only features relevant to modeling and removed records with missing values to ensure a consistent and complete dataset.

| Feature | Description |
|:-------:|:-----------:|
| SongID | Unique Identifier (concatenation of concatenation of song and performer) |
| Performer | The name of the performer or artist of the song. (Text) |
| Song | The title of the song. (Text) |
| spotify_genre | The genre(s) of the song according to Spotify's classification system. (Text) |
| spotify_track_duration_ms | The duration of the song in milliseconds. (Numeric) |
| spotify_track_explicit | Indicates whether the song contains explicit content or not. (0 or 1) |
| danceability | A measurement criteria combining musical elements in terms of suitability for dancing. (Numeric) |
| energy | Characterizes the intensity and activity within each recording. (Numeric) |
| key | Showcases tonalities such as C major or D minor described using text representations. (Text) |
| loudness | Expressed as decibel levels measuring the overall volume across songs. (Numeric) |
| mode | Distinguishes major or minor tonalities indicated as numeric values within this context. (Numeric) |
| speechiness | Shows the presence of spoken-word elements quantitatively within tracks. (Numeric) |
| acousticness | Displays the acoustic qualities of songs numerically, reflecting the contrast between natural sound and electronically enhanced production or elements. (Numeric) |
| instrumentalness | Quantifies the likelihood of a song being instrumental using numeric metrics. (Numeric) |
| liveness | Reveals the presence of an audience within live recordings via numerical evaluation. (Numeric)  |
| valence | Describes the musical positivity conveyed by songs quantitatively as numeric measurements on a scale. (Numeric) |
| tempo | Measures the beats per minute (BPM) for tempo indication evaluated by numeric means. (Numeric) |
| time_signature | Specifies the song structure like 4/4 or 3/4 using text representation. (Text) |

In [88]:
# select only audio features relevant for analysis and modeling
relevant_audio_features = [
    "SongID",
    "spotify_genre",
    "spotify_track_duration_ms",
    "spotify_track_explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

# subset the uncleaned dataset to include only the selected features
audio = audio[relevant_audio_features]

# remove rows with missing values to ensure a complete feature set
audio = audio.dropna()

## Feature Representation

### Convert Booleans to 0/1

In [89]:
# convert boolean to 0 and 1
audio["spotify_track_explicit"] = audio["spotify_track_explicit"].astype(int)

### Embedding Genres

In [90]:
# replace empty [] genre values with ["unknown"] for consistency
def normalize_genres(x):
    if x == "[]":
        return ["unknown"]
    return ast.literal_eval(x)

# apply to audio df
audio["spotify_genre"] = audio["spotify_genre"].apply(normalize_genres)

In [91]:
# Collect unique genres
all_genres = sorted({g for genres in audio["spotify_genre"] for g in genres})

# Reserve special tokens for 
# 1) padding 
# 2) and unknown/new genres in unseen songs (testing) 
PAD = "<PAD>"
UNK = "<UNK>"

# initialize a dictionary mapping of genres, starting from 2
genre2idx = {PAD: 0, UNK: 1}
for g in all_genres:
    if g not in genre2idx:
        genre2idx[g] = len(genre2idx)

# reverse mapping, from index to genre string.
idx2genre = {i: g for g, i in genre2idx.items()}

# counts the number of unique genres
vocab_size = len(genre2idx)

print(
    f"Genre vocabulary size: {vocab_size:,}\n"
    f"Example genres: {', '.join(list(genre2idx.keys())[2:7])}"
)

Genre vocabulary size: 1,045
Example genres: a cappella, acid house, acid jazz, acoustic blues, acoustic pop


In [92]:
# set limit on the number of genres per song
MAX_GENRES = 5

# function to encode string genre values to integer ID
def encode_genres(genres):
    # get ids from genre2idx, falls back to <unk> if not seen.
    ids = [genre2idx.get(g, genre2idx[UNK]) for g in genres]

    # truncate
    ids = ids[:MAX_GENRES]  
    # pad with 0 if less than 5 genres
    ids += [genre2idx[PAD]] * (MAX_GENRES - len(ids))  

    return ids

# apply to audio dataset
audio["spotify_genre_ids"] = audio["spotify_genre"].apply(encode_genres)

audio.head(5)

,SongID,spotify_genre,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,spotify_genre_ids
2,......And Roses And RosesAndy Williams,"[adult standards, brill building pop, easy lis...",166106.0,0,0.154,0.185,5.0,-14.063,1.0,0.0315,0.91100,0.000267,0.112,0.150,83.969,4.0,"[8, 127, 365, 614, 0]"
3,...And Then There Were DrumsSandy Nelson,"[rock-and-roll, space age pop, surf music]",172066.0,0,0.588,0.672,11.0,-17.278,0.0,0.0361,0.00256,0.745000,0.145,0.801,121.962,4.0,"[844, 910, 926, 0, 0]"
4,...Baby One More TimeBritney Spears,"[dance pop, pop, post-teen pop]",211066.0,0,0.759,0.699,0.0,-5.745,0.0,0.0307,0.20200,0.000131,0.443,0.907,92.960,4.0,"[277, 764, 785, 0, 0]"
5,...Ready For It?Taylor Swift,"[pop, post-teen pop]",208186.0,0,0.613,0.764,2.0,-6.509,1.0,0.1360,0.05270,0.000000,0.197,0.417,160.015,4.0,"[764, 785, 0, 0, 0]"
7,'65 Love AffairPaul Davis,"[album rock, bubblegum pop, country rock, folk...",219813.0,0,0.647,0.686,2.0,-4.247,0.0,0.0274,0.43200,0.000006,0.133,0.952,155.697,4.0,"[16, 145, 268, 409, 614]"


In [93]:
# Convert the encoded genre-id lists in the dataframe to a (N, MAX_GENRES) tensor
genre_ids = torch.tensor(
    pd.DataFrame(audio["spotify_genre_ids"].to_list()).values,
    dtype=torch.long
)  # (N, MAX_GENRES)

In [94]:
class GenreEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

    def forward(self, genre_ids):
        """
        genre_ids: (batch, MAX_GENRES) long
        returns: (batch, embed_dim) float
        """
        e = self.emb(genre_ids)  # (batch, MAX_GENRES, embed_dim)

        # Mask out PAD tokens so they don't affect the pooled embedding
        mask = (genre_ids != self.pad_idx).unsqueeze(-1).float()  # (batch, MAX_GENRES, 1)
        e = e * mask

        denom = mask.sum(dim=1).clamp(min=1.0)  # (batch, 1)
        pooled = e.sum(dim=1) / denom          # (batch, embed_dim)
        return pooled

# Initialize encoder using your vocab size and PAD index
genre_encoder = GenreEncoder(vocab_size=vocab_size, embed_dim=16, pad_idx=genre2idx[PAD])

# Encode genres into vectors
genre_vecs = genre_encoder(genre_ids)  # (N, 16)

genre_vecs.shape

torch.Size([24186, 16])

In [95]:
# embedding dim (16, but can be adjusted)
embed_dim = genre_vecs.shape[1]

# convert the genre embedding tensor back to a pandas DataFrame
genre_vec_df = pd.DataFrame(
    # chat told me to add this in case anyone using GPU
    genre_vecs.detach().cpu().numpy(), 
    columns=[f"genre_emb_{i+1}" for i in range(embed_dim)],
    index=audio.index
)

# preview first few rows of the genre embedding
genre_vec_df.head(5)

,genre_emb_1,genre_emb_2,genre_emb_3,genre_emb_4,genre_emb_5,genre_emb_6,genre_emb_7,genre_emb_8,genre_emb_9,genre_emb_10,genre_emb_11,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16
2,0.171244,-0.306219,0.030578,0.666900,-0.424515,0.171154,0.131719,0.045151,0.427118,-0.121575,-0.158559,0.180203,0.405925,0.575688,0.792574,-0.137528
3,-0.179796,0.368559,-0.099204,0.570811,-0.116604,0.604052,-0.022766,0.445687,-0.500438,-0.429720,-0.318278,0.775808,0.063394,0.145840,0.237928,-0.139751
4,-1.217189,-0.106377,-1.078668,-0.949431,1.001791,0.462621,-0.307503,-0.354504,-0.219591,-0.472916,0.470733,-0.621630,0.524550,0.087097,0.641945,0.347665
5,-1.267263,0.424386,-0.796611,-1.003890,0.538342,0.984852,-0.148557,-0.604630,0.035902,-0.210625,-0.013394,-0.744986,0.219712,0.544104,0.401829,1.138462
7,-0.128825,-0.155267,0.260334,0.718790,-0.273618,0.058279,0.528503,0.145543,-0.255186,-0.392838,0.191883,-0.293151,-0.095506,0.639190,-0.096971,-0.650049


In [96]:
# attach embeddings to audio df
audio = pd.concat([audio, genre_vec_df], axis=1)

# drop old genre columns
audio = audio.drop(columns=["spotify_genre", "spotify_genre_ids"])

# columns 
print(audio.columns)

Index(['SongID', 'spotify_track_duration_ms', 'spotify_track_explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'genre_emb_1', 'genre_emb_2', 'genre_emb_3',
       'genre_emb_4', 'genre_emb_5', 'genre_emb_6', 'genre_emb_7',
       'genre_emb_8', 'genre_emb_9', 'genre_emb_10', 'genre_emb_11',
       'genre_emb_12', 'genre_emb_13', 'genre_emb_14', 'genre_emb_15',
       'genre_emb_16'],
      dtype='object')


## Merge Datasets

In [97]:
# merge the cleaned audio features with aggregated chart data using SongID
df = audio.merge(lb_song, on="SongID", how="inner")

# sort df by first week 
df = df.sort_values(by="first_week")

df.columns

Index(['SongID', 'spotify_track_duration_ms', 'spotify_track_explicit',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'genre_emb_1', 'genre_emb_2', 'genre_emb_3',
       'genre_emb_4', 'genre_emb_5', 'genre_emb_6', 'genre_emb_7',
       'genre_emb_8', 'genre_emb_9', 'genre_emb_10', 'genre_emb_11',
       'genre_emb_12', 'genre_emb_13', 'genre_emb_14', 'genre_emb_15',
       'genre_emb_16', 'first_week', 'last_week', 'peak_position',
       'weeks_on_chart', 'hit'],
      dtype='object')

## Defining what a "Hit" is?

We defined a hit as a song that reached a Top 10 peak position and remained on the chart for at least 15 weeks.

$$
\text{Hit} = \text{Weeks on Chart} \ge 15 \ \text{\&} \ \text{Peak Position} \le 10
$$

This definition yields 3,457 songs, or approximately 14.3% of the 24,179 songs in the dataset, producing a hit class that is intentionally selective. 

In [98]:
hit_percentage = df["hit"].mean() * 100
hit_percentage

14.30225346289022

Defining hits as roughly the top 15% of charting songs balances exclusivity with sufficient sample size. 
- A more restrictive definition (e.g., Top 5 or fewer weeks on the chart) would yield too few songs, reducing statistical power and overemphasizing extreme outliers. 

- Conversely, a more general definition (e.g., Top 40 or brief chart appearances) would dilute the concept of a hit by including songs with limited impact. This threshold therefore captures meaningful commercial success while maintaining analytical robustness.

In [99]:
# first week in the complete df
min_week = df["first_week"].min()

# last week in the complete df
max_week = df["last_week"].max()

print(f"Date range covered by the dataset: {min_week:%B %d, %Y} to {max_week:%B %d, %Y}")
print(f"Number of rows in the cleaned dataset: {len(df):,}")

Date range covered by the dataset: August 02, 1958 to May 29, 2021
Number of rows in the cleaned dataset: 24,185


In [100]:
# sort the cleaned dataset from earliest date to latest
df = df.sort_values(by="first_week")

In [101]:
# spot check begining 
df.head(5)

,SongID,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16,first_week,last_week,peak_position,weeks_on_chart,hit
10923,JudyFrankie Vaughan,125933.0,0,0.517,0.420,9.0,-12.751,0.0,0.0713,0.964,...,1.228887,-0.910680,-0.904488,1.281385,-0.482246,1958-08-02,1958-08-02,100,1,0
343,A Certain SmileJohnny Mathis,168293.0,0,0.233,0.337,5.0,-10.031,1.0,0.0307,0.854,...,0.617252,-0.142712,0.559327,0.334065,-0.067055,1958-08-02,1958-09-27,22,9,0
10977,Just A DreamJimmy Clanton And His Rockets,152973.0,0,0.610,0.326,7.0,-12.266,1.0,0.0505,0.687,...,-1.330139,0.024099,1.565437,1.757846,0.303156,1958-08-02,1958-11-08,4,15,1
18936,Summertime BluesEddie Cochran,119360.0,0,0.715,0.882,11.0,-8.610,0.0,0.0593,0.123,...,-0.024910,-0.181583,0.034302,0.831113,-0.020146,1958-08-02,1958-11-15,8,16,1
7949,High School ConfidentialJerry Lee Lewis And Hi...,150026.0,0,0.608,0.920,10.0,-6.792,1.0,0.0422,0.714,...,-1.330139,0.024099,1.565437,1.757846,0.303156,1958-08-02,1958-08-09,63,2,0


In [102]:
# spot check end
df.tail(5)

,SongID,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16,first_week,last_week,peak_position,weeks_on_chart,hit
6220,FractionsNicki Minaj,181690.0,1,0.924,0.523,0.0,-6.362,1.0,0.1320,0.0110,...,-0.941837,0.686496,0.080765,-0.026761,0.107769,2021-05-29,2021-05-29,52,1,0
22943,White TeethYoungBoy Never Broke Again,174683.0,1,0.638,0.660,3.0,-7.115,0.0,0.2990,0.2460,...,0.174432,0.847476,-0.195007,0.996027,0.753898,2021-05-29,2021-05-29,78,1,0
18740,StraighteninMigos,255532.0,1,0.847,0.629,9.0,-5.810,1.0,0.1020,0.0318,...,0.163346,0.384727,0.149587,-0.513568,-0.106328,2021-05-29,2021-05-29,38,1,0
20739,Things A Man Oughta KnowLainey Wilson,203373.0,0,0.659,0.683,3.0,-5.623,1.0,0.0312,0.5130,...,0.776841,-1.334719,-0.276970,0.045458,0.747542,2021-05-29,2021-05-29,94,1,0
2708,Build A BitchBella Poarch,122772.0,1,0.855,0.463,3.0,-7.454,1.0,0.0367,0.2170,...,-1.330139,0.024099,1.565437,1.757846,0.303156,2021-05-29,2021-05-29,58,1,0


In [103]:
# final spot check
df.isnull().sum()

SongID                       0
spotify_track_duration_ms    0
spotify_track_explicit       0
danceability                 0
energy                       0
key                          0
loudness                     0
mode                         0
speechiness                  0
acousticness                 0
instrumentalness             0
liveness                     0
valence                      0
tempo                        0
time_signature               0
genre_emb_1                  0
genre_emb_2                  0
genre_emb_3                  0
genre_emb_4                  0
genre_emb_5                  0
genre_emb_6                  0
genre_emb_7                  0
genre_emb_8                  0
genre_emb_9                  0
genre_emb_10                 0
genre_emb_11                 0
genre_emb_12                 0
genre_emb_13                 0
genre_emb_14                 0
genre_emb_15                 0
genre_emb_16                 0
first_week                   0
last_wee

---

## Random Train-test Split

In [104]:
# separate features and target
X = df.drop(columns=["SongID", "hit", "peak_position", "weeks_on_chart", "first_week", "last_week"])
y = df["hit"]

# train-test split (80% train, 20% test) roughly 5000 test and 19000 train
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=426,

    # preserves hit / non-hit ratio to ensure train and set sets have 
    # similar class balance
    stratify= y,

    shuffle=True
)

print(f"Training Set:   n = {len(X_train)} ({len(X_train)/24185:.2f}%)" +
       f"\nTesting Set:    n = {len(X_test)} ({len(X_test)/24185:.2f}%)")

Training Set:   n = 19348 (0.80%)
Testing Set:    n = 4837 (0.20%)


## Time-based Train-test Split

### Rationale:

We are claiming the model learns temporal/stylistic structure that generalizes. Only a forward-in-time test actually measures that.

Doing a time-based train-test split also avoids inflated performance. A random split mixes eras, so the model can exploit era-correlated cues while still being evaluated on the same distribution. So the model can still look great even if it fails on genuinely “future” styles.

In [105]:
# # cut-offs for training, validation, and test sets
# train_end = pd.Timestamp("2009-12-31")
# val_end = pd.Timestamp("2015-12-31")
# test_end = pd.Timestamp("2021-05-29")

# # create boolean 
# train_mask = df["first_week"] <= train_end
# val_mask   = (df["first_week"] > train_end) & (df["first_week"] <= val_end)
# test_mask  = (df["first_week"] > val_end) & (df["first_week"] <= test_end)

# # separate features and target
# X = df.drop(columns=["SongID", "hit", "peak_position", "weeks_on_chart"])
# y = df["hit"]

# # split X/y (NO shuffle, NO stratify; time split defines it)
# X_train, y_train = X.loc[train_mask], y.loc[train_mask]
# X_val,   y_val   = X.loc[val_mask],   y.loc[val_mask]
# X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

# size of datasets
# print(f"Training Set:   n = {len(X_train)} ({len(X_train)/24185:.2f}%)" +
#        f"\nValidation Set: n = {len(X_val)} ({len(X_val)/24185:.2f}%)" +
#        f"\nTesting Set:    n = {len(X_test)} ({len(X_test)/24185:.2f}%)")

## Save to CSV Files

Make sure to comment out the code for the train-split you don't want.

In [106]:
# save df into csv files 
train_df = pd.concat([X_train, y_train], axis=1)
# val_df   = pd.concat([X_val, y_val], axis=1)
test_df  = pd.concat([X_test, y_test], axis=1)

train_df.to_csv("dataset/train.csv", index=False)
# val_df.to_csv("dataset/val.csv", index=False)
test_df.to_csv("dataset/test.csv", index=False)